# Exploring TfL Bikepoint API

In this notebook we will be attempting to hit the TfL Bikepoint API to get an understanding of the kind of data we get, what it looks like, and propose a potential schema for how data coming from this API will be represented

The main API we will be hitting is the BikePoint API, which has the following description, pulled from TfL Website: 

Gets all bike point locations. The `Place` object has an `addtionalProperties` array which contains the `nbBikes`, `nbDocks` and `nbSpaces` numbers which give the status of the `BikePoint`. A mismatch in these numbers i.e. `nbDocks - (nbBikes + nbSpaces) != 0` indicates broken docks.

In [7]:
import requests
import pandas as pd
import pydantic
from datetime import datetime

In [3]:
ENDPOINT_URL = "https://api.tfl.gov.uk/BikePoint/"

# Send the request
response = requests.get(ENDPOINT_URL)

# Check if successful (Status code 200)
if response.status_code == 200:
    # Parse the response data as JSON
    data = response.json()
else:
    print(f"Error: {response.status_code}")

In [4]:
len(data)

797

Ok so the number of places is quite large (approx 800) let's inspect the structure of a single place

In [33]:
data[2]

{'$type': 'Tfl.Api.Presentation.Entities.Place, Tfl.Api.Presentation.Entities',
 'id': 'BikePoints_3',
 'url': '/Place/BikePoints_3',
 'commonName': 'Christopher Street, Liverpool Street',
 'placeType': 'BikePoint',
 'additionalProperties': [{'$type': 'Tfl.Api.Presentation.Entities.AdditionalProperties, Tfl.Api.Presentation.Entities',
   'category': 'Description',
   'key': 'TerminalName',
   'sourceSystemKey': 'BikePoints',
   'value': '001012',
   'modified': '2026-05-26T18:41:39.257Z'},
  {'$type': 'Tfl.Api.Presentation.Entities.AdditionalProperties, Tfl.Api.Presentation.Entities',
   'category': 'Description',
   'key': 'Installed',
   'sourceSystemKey': 'BikePoints',
   'value': 'true',
   'modified': '2026-05-26T18:41:39.257Z'},
  {'$type': 'Tfl.Api.Presentation.Entities.AdditionalProperties, Tfl.Api.Presentation.Entities',
   'category': 'Description',
   'key': 'Locked',
   'sourceSystemKey': 'BikePoints',
   'value': 'false',
   'modified': '2026-05-26T18:41:39.257Z'},
  {'$ty

Ok so the salient properties from this JSON are going to be 

- place id
- the bike point name
- latitude
- longitude
- additional attribute where key is NbBikes
- additional attribute where key is NbEmptyDocks
- additional attribute where key is NbDocks
- additional attribute where key is NbStandardBikes
- additional attribute where key is NbEBikes

Although I suppose from a data engineering standpoint, we shouldn't be having this logic stored on our end. if TfL adds a new endpoint that turns out to be salient for the forecasting model, this would discard it. Let's just flatten out `additionalAttributes`

In [45]:
from datetime import datetime

def convert_str_datetime_to_epoch(str_datetime: str) -> int:
    dt = datetime.fromisoformat(str_datetime.replace('Z', '+00:00'))
    timestamp = dt.timestamp()
    return int(timestamp)
    
raw_data_point = data[0]
refined_data_point = dict()

refined_data_point["id"] = raw_data_point['id']
refined_data_point["name"] = raw_data_point['commonName']
refined_data_point["lat"] = raw_data_point["lat"]
refined_data_point["long"] = raw_data_point["lon"]

additional_attributes = raw_data_point['additionalProperties']

for attribute in additional_attributes:
    raw_attribute_key = attribute['key']
    print(f"Found additional attribute {raw_attribute_key}")

    refined_attribute_key = raw_attribute_key[0].lower() + raw_attribute_key[1:]
    refined_data_point[refined_attribute_key] = attribute["value"]

    if "timestamp" not in refined_data_point:
        refined_data_point["timestamp"] = convert_str_datetime_to_epoch(attribute["modified"])

Found additional attribute TerminalName
Found additional attribute Installed
Found additional attribute Locked
Found additional attribute InstallDate
Found additional attribute RemovalDate
Found additional attribute Temporary
Found additional attribute NbBikes
Found additional attribute NbEmptyDocks
Found additional attribute NbDocks
Found additional attribute NbStandardBikes
Found additional attribute NbEBikes


In [46]:
refined_data_point

{'id': 'BikePoints_1',
 'name': 'River Street , Clerkenwell',
 'lat': 51.529163,
 'long': -0.10997,
 'terminalName': '001023',
 'timestamp': 1779825007,
 'installed': 'true',
 'locked': 'false',
 'installDate': '1278947280000',
 'removalDate': '',
 'temporary': 'false',
 'nbBikes': '4',
 'nbEmptyDocks': '12',
 'nbDocks': '19',
 'nbStandardBikes': '4',
 'nbEBikes': '0'}

Hmm okay, this looks decent for a first cut of the forecasting model. Let us get this to iterate over our whole dataset and create our dataframe

In [8]:
full_refined_dataset = list()

def convert_str_datetime_to_epoch(str_datetime: str) -> int:
    dt = datetime.fromisoformat(str_datetime.replace('Z', '+00:00'))
    timestamp = dt.timestamp()
    return int(timestamp)

def filter_additional_attributes(raw_additional_attributes: dict, refined_dict: dict) -> None:
    for attribute in raw_additional_attributes:
        raw_attribute_key = attribute['key']
        refined_key = raw_attribute_key[0].lower() + raw_attribute_key[1:]
        refined_dict[refined_key] = attribute["value"] if len(attribute["value"]) > 0 else None

        if "timestamp" not in refined_dict:
            refined_dict["timestamp"] = convert_str_datetime_to_epoch(attribute["modified"])

def convert_raw_to_refined(raw_data: dict) -> dict:
    refined_data_point = dict()
    refined_data_point["id"] = raw_data['id']
    refined_data_point["name"] = raw_data['commonName']
    refined_data_point["lat"] = raw_data["lat"]
    refined_data_point["long"] = raw_data["lon"]
    
    additional_attributes = raw_data['additionalProperties']

    filter_additional_attributes(additional_attributes, refined_data_point)

    return refined_data_point

for data_point in data:
    full_refined_dataset.append(convert_raw_to_refined(data_point))

df = pd.DataFrame.from_dict(full_refined_dataset)

Okay! I think this looks pretty good as our supabase table schema, and additional for our pydantic class we will have the same concepts encodeded in the object

In [71]:
from pydantic import BaseModel 

class BikePoint(BaseModel):
    id: str
    name: str
    lat: float
    long: float
    terminalName: int
    timestamp: int
    installed: bool
    locked: bool
    installDate: int
    removalDate: float
    temporary: bool
    nbBikes: int
    nbEmptyDocks: int
    nbDocks: int
    nbStandardBikes: int 
    nbEBikes: int

In [9]:
df.head()

,id,name,lat,long,terminalName,timestamp,installed,locked,installDate,removalDate,temporary,nbBikes,nbEmptyDocks,nbDocks,nbStandardBikes,nbEBikes
0,BikePoints_1,"River Street , Clerkenwell",51.529163,-0.109970,001023,1779825007,true,false,1278947280000,NaN,false,4,12,19,4,0
1,BikePoints_2,"Phillimore Gardens, Kensington",51.499606,-0.197574,001018,1779830340,true,false,1278585780000,NaN,false,1,33,37,0,1
2,BikePoints_3,"Christopher Street, Liverpool Street",51.521283,-0.084605,001012,1779820899,true,false,1278240360000,NaN,false,0,32,32,0,0
3,BikePoints_4,"St. Chad's Street, King's Cross",51.530059,-0.120973,001013,1779831175,true,false,1278241080000,NaN,false,7,14,23,7,0
4,BikePoints_5,"Sedding Street, Sloane Square",51.493130,-0.156876,003420,1779827425,true,false,1278241440000,NaN,false,0,20,27,0,0


In [70]:
BikePoint(**df.iloc[0].to_dict())

BikePoint(id='BikePoints_1', name='River Street , Clerkenwell', lat=51.529163, long=-0.10997, terminalName=1023, timestamp=1779825007, installed=True, locked=False, installDate=1278947280000, removalDate=nan, temporary=False, nbBikes=4, nbEmptyDocks=12, nbDocks=19, nbStandardBikes=4, nbEBikes=0)